# Ceres Raw Integrity, Repair, And Kaggle Shard Workflow

Colab workflow for resumable raw GeoTIFF integrity scans, repair candidate generation, repair verification, and Kaggle shard planning.

In [ ]:
from __future__ import annotations

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
RAW_DIR = DRIVE_ROOT / "wheat_data_v2.0-beta"
REPORT_ROOT = DRIVE_ROOT / "Ceres" / "raw_integrity_reports" / "wheat_data_v2.0-beta"
REPAIR_ROOT = DRIVE_ROOT / "Ceres" / "raw_repairs"
BATCH_SIZE = 100
READ_SAMPLE = True
WEEK_FAILURE_THRESHOLD = 5
KAGGLE_SHARD_MAX_GB = 150.0
KAGGLE_SLUG_PREFIX = "ceres-raw"


## Mount Google Drive

Run this in Colab before scanning.

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Colab; skipping Drive mount.")


## Setup Helpers

Install runtime packages if needed, then import repository helpers.

In [ ]:
import csv
import os
import subprocess
import sys

CERES_REPO_URL = "https://github.com/AgriNova-Orbital/ceres-ai-pipeline.git"
CERES_REPO_REF = "main"
REPO_DIR = Path("/content/ceres-ai-pipeline")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio", "tqdm"])
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", CERES_REPO_URL, str(REPO_DIR)])
subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", CERES_REPO_REF])
subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "FETCH_HEAD"])
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

from modules.wheat_risk.raw_integrity import (
    build_curated_manifest,
    build_repair_candidates,
    choose_batch,
    load_scan_records,
    next_batch_path,
    plan_kaggle_shards,
    scan_raw_batch,
    write_dict_csv,
    write_scan_csv,
    write_summary_json,
)


## Batch Raw Integrity Scan

Each run scans one batch and skips files already present in completed batch CSVs.

In [ ]:
batch_dir = REPORT_ROOT / "batches"
completed = set()
for csv_path in sorted(batch_dir.glob("batch_*.csv")):
    with csv_path.open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            completed.add(row["relative_path"])

candidates = sorted(RAW_DIR.rglob("*.tif"))
selected = choose_batch(
    candidates,
    raw_root=RAW_DIR,
    completed_relative_paths=completed,
    batch_size=BATCH_SIZE,
)
records = scan_raw_batch(selected, raw_root=RAW_DIR, read_sample=READ_SAMPLE)
batch_path = next_batch_path(batch_dir)
write_scan_csv(batch_path, records)
all_records = load_scan_records(batch_dir)
write_summary_json(REPORT_ROOT / "summary.json", all_records)
repair_candidates = build_repair_candidates(all_records, week_failure_threshold=WEEK_FAILURE_THRESHOLD)
write_dict_csv(
    REPORT_ROOT / "repair_candidates.csv",
    repair_candidates,
    columns=("repair_scope", "week_key", "relative_path", "tile_key", "reason"),
)
print(f"Scanned {len(records)} files into {batch_path}")


## Earth Engine repair export

Use `repair_candidates.csv` to submit targeted Earth Engine repair exports. Write repaired files under `REPAIR_ROOT / week_key / run_id` and never overwrite `RAW_DIR`. Record export task IDs and expected output paths beside the repair output.

## Repair Verification And Curated Manifest

After repair exports finish, scan the repair folder with the same scanner. Build `repair_replacements.csv`, then use verified `ok` repair records to create `curated_manifest.csv`.

In [ ]:
# Example shape for a verified replacement map after repair verification.
# Fill this from repair_replacements.csv once repair outputs have status == ok.
# all_records = load_scan_records(REPORT_ROOT / "batches")
replacement_map = {}
repair_records = []
# curated = build_curated_manifest(all_records, repair_records=repair_records, replacements=replacement_map)
# write_dict_csv(REPORT_ROOT / "curated_manifest.csv", curated, columns=("original_relative_path", "source_relative_path", "source_status", "size_bytes", "week_key"))


## Kaggle Shard Planning

Plan raw dataset shards only after `curated_manifest.csv` has no unresolved bad files. Keep Kaggle credentials outside this notebook.

In [ ]:
# Example shard planning call once curated rows are loaded from curated_manifest.csv.
# shards = plan_kaggle_shards(curated, max_shard_bytes=int(KAGGLE_SHARD_MAX_GB * 1024 * 1024 * 1024), slug_prefix=KAGGLE_SLUG_PREFIX)
# write_dict_csv(REPORT_ROOT / "kaggle_shards.csv", shards, columns=("dataset_slug", "part_number", "file_count", "total_bytes", "week_keys"))
